In [ ]:
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# =========================
# 1. ĐƯỜNG DẪN FILE
# =========================
file_input = r"D:\py\KhaiPhaDlWeb\DeAn1\imdb_cũ\DuLieu\imdb_movies_2020.xlsx"      # file Excel đầu vào, có cột Link
file_output = r'D:\py\KhaiPhaDlWeb\DeAn1\imdb_dataset_2020.xlsx'   # file Excel đầu ra

# =========================
# 2. HEADERS
# =========================
# Dùng Accept-Language = en để IMDb trả về nhãn như Director, Stars...
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

session = requests.Session()
session.headers.update(HEADERS)

# =========================
# 3. CÁC CỘT KẾT QUẢ
# =========================
COT_KQ = [
    "Movie_Title",
    "Released_Date",
    "Runtime",
    "Genre",
    "No_of_votes",
    "Director",
    "Stars",
    "Gross",
    "Countries_of_origin"
]

def TaoDongRong():
    return {cot: "" for cot in COT_KQ}

def LayText(tag):
    if tag:
        return tag.get_text(" ", strip=True)
    return ""

def TimCotLink(df):
    for col in df.columns:
        if str(col).strip().lower() == "link":
            return col
    raise ValueError("Không tìm thấy cột Link trong file Excel.")

def LayMucTheoTestID(soup, testid):
    return soup.find("li", attrs={"data-testid": testid})

def LayDanhSachTextTrongItem(item):
    ds = []
    if not item:
        return ds

    tags = item.select("div.ipc-metadata-list-item__content-container li.ipc-inline-list__item")
    for tag in tags:
        text = tag.get_text(" ", strip=True)
        if text:
            ds.append(text)

    return ds

def LayCreditTheoNhan(soup, danh_sach_nhan):
    """
    Vì Director và Stars đều có cùng data-testid = title-pc-principal-credit,
    nên phải so sánh theo nhãn (Director / Directors / Star / Stars)
    """
    items = soup.find_all("li", attrs={"data-testid": "title-pc-principal-credit"})
    danh_sach_nhan = [x.lower() for x in danh_sach_nhan]

    for item in items:
        label_tag = item.find(["span", "a"], class_="ipc-metadata-list-item__label")
        label_text = LayText(label_tag).lower()

        if label_text in danh_sach_nhan:
            return item

    return None

def LayTenPhim(soup):
    tag = soup.find("span", attrs={"data-testid": "hero__primary-text"})
    return LayText(tag)

def LayDaoDien(soup):
    item = LayCreditTheoNhan(soup, ["Director", "Directors"])
    ds = LayDanhSachTextTrongItem(item)
    return " | ".join(ds)

def LayDienVien(soup):
    item = LayCreditTheoNhan(soup, ["Star", "Stars"])
    ds = LayDanhSachTextTrongItem(item)
    return " | ".join(ds)

import json

def LayTheLoai(soup):
    try:
        # Ưu tiên lấy từ JSON-LD của IMDb
        scripts = soup.find_all("script", type="application/ld+json")
        for script in scripts:
            try:
                if not script.string:
                    continue

                data = json.loads(script.string)

                # Có trang IMDb để dict
                if isinstance(data, dict) and data.get("@type") == "Movie":
                    genre = data.get("genre", "")
                    if isinstance(genre, list):
                        return "|".join(genre)
                    elif isinstance(genre, str):
                        return genre

                # Có trang IMDb để list
                if isinstance(data, list):
                    for item in data:
                        if isinstance(item, dict) and item.get("@type") == "Movie":
                            genre = item.get("genre", "")
                            if isinstance(genre, list):
                                return "|".join(genre)
                            elif isinstance(genre, str):
                                return genre
            except:
                pass

        # Nếu không có trong JSON thì mới fallback sang HTML cũ
        item = soup.find("li", attrs={"data-testid": "storyline-genres"})
        if item:
            genres = []
            for a in item.find_all("a", href=True):
                text = a.get_text(strip=True)
                if text:
                    genres.append(text)
            return "|".join(genres)

        return ""
    except:
        return ""

def LayNgayPhatHanh(soup):
    item = LayMucTheoTestID(soup, "title-details-releasedate")
    if not item:
        return ""

    tag = item.select_one("div.ipc-metadata-list-item__content-container li.ipc-inline-list__item")
    return LayText(tag)

def LayRuntime(soup):
    item = LayMucTheoTestID(soup, "title-techspec_runtime")
    if not item:
        return ""

    tag = item.select_one("div.ipc-metadata-list-item__content-container span.ipc-metadata-list-item__list-content-item")
    return LayText(tag)

def LaySoNguoiDanhGia(soup):
    tag = soup.find("span", attrs={"data-testid": "rating-histogram-vote-count"})
    return LayText(tag)

def LayGross(soup):
    item = LayMucTheoTestID(soup, "title-boxoffice-cumulativeworldwidegross")
    if not item:
        return ""

    tag = item.select_one("div.ipc-metadata-list-item__content-container span.ipc-metadata-list-item__list-content-item")
    return LayText(tag)

# ---------- HÀM PHỤ, CHƯA ĐƯA VÀO DATASET CUỐI ----------
# Vì bạn có hỏi cả Countries of origin, mình viết sẵn luôn.
# Hiện chưa ghi vào file vì danh sách cột cuối cùng của bạn không có trường này.
def LayQuocGia(soup):
    item = LayMucTheoTestID(soup, "title-details-origin")
    ds = []

    if item:
        tags_a = item.select("div.ipc-metadata-list-item__content-container a")
        for a in tags_a:
            text = a.get_text(strip=True)
            if text:
                ds.append(text)

    return "|".join(ds)

def LayThongTinPhim(url):
    dong = TaoDongRong()

    try:
        response = session.get(url, timeout=20)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        dong["Movie_Title"] = LayTenPhim(soup)
        dong["Released_Date"] = LayNgayPhatHanh(soup)
        dong["Runtime"] = LayRuntime(soup)
        dong["Genre"] = LayTheLoai(soup)
        dong["No_of_votes"] = LaySoNguoiDanhGia(soup)
        dong["Director"] = LayDaoDien(soup)
        dong["Stars"] = LayDienVien(soup)
        dong["Gross"] = LayGross(soup)

        # Nếu muốn xem thử quốc gia:
        dong["Countries_of_origin"] = LayQuocGia(soup)
        # print("Countries:", quoc_gia)

    except Exception as e:
        print(f"Lỗi khi đọc link: {url}")
        print("Chi tiết lỗi:", e)
        # Giữ nguyên dòng rỗng nếu lỗi hoặc thiếu dữ liệu

    return dong

# =========================
# 4. ĐỌC EXCEL VÀ CÀO DỮ LIỆU
# =========================
df = pd.read_excel(file_input)
cot_link = TimCotLink(df)

ket_qua = []

for i, link in enumerate(df[cot_link], start=1):
    url = "" if pd.isna(link) else str(link).strip()
    print(f"Đang xử lý dòng {i}: {url}")

    if url == "":
        ket_qua.append(TaoDongRong())
        continue

    thong_tin = LayThongTinPhim(url)
    ket_qua.append(thong_tin)

    # Nghỉ 1 giây để tránh gửi request quá nhanh
    time.sleep(1)

# =========================
# 5. LƯU FILE EXCEL KẾT QUẢ
# =========================
df_kq = pd.DataFrame(ket_qua, columns=COT_KQ)
df_kq.to_excel(file_output, index=False)

print("Đã lưu xong file:", file_output)